In [3]:
import json

with open("RESULTS/training_50_EneSou_BodOfWat_Org_PhyPhe_Loc_PhyArt_NatDis_Che_BodPar_MatExp_GeoFea_Org_IntArt_Sys_Ass_Met_FieOfStu_MetPhe_TimPer_Eco_Pol_NatPhe_Qua_Per_Dis_MeaDev_Sat.json") as f:
    json_file = json.load(f)
    
with open("RESULTS/golden_50_EneSou_BodOfWat_Org_PhyPhe_Loc_PhyArt_NatDis_Che_BodPar_MatExp_GeoFea_Org_IntArt_Sys_Ass_Met_FieOfStu_MetPhe_TimPer_Eco_Pol_NatPhe_Qua_Per_Dis_MeaDev_Sat.json") as f:
    json_file_g = json.load(f)

In [6]:
print(len(json_file))
print(len(json_file_g))
print(len(json_file) + len(json_file_g))

1027
192
1219


In [1]:
# 3. Load Data
from datasets import load_dataset
dataset_id = "P0L3/CliReNER_v_1_1_28_SILVER"

print(f"Loading dataset: {dataset_id}")
dataset = load_dataset(dataset_id)

# Extract labels (Assumes standard NER format)
# labels = dataset["train"].features["ner_tags"][0].names
labels = dataset["train"].features["ner_tags"].feature.names


Loading dataset: P0L3/CliReNER_v_1_1_28_SILVER


In [6]:
total = 0
for key in dataset:
    print(len(dataset[key]))
    total += len(dataset[key])
    
print(total)

803
106
118
1027


In [11]:
import seaborn as sns

sns.color_palette("Oranges")
print(sns.color_palette("Oranges").as_hex())

['#fee3c8', '#fdc692', '#fda057', '#f67824', '#e05206', '#ad3803']


In [7]:
from dataset_processing import cwed4eta_process_json_file, transform_to_ner_format, CLIRENER_LABELS_V1, ner_dataset_to_hf_format
import argparse
import pathlib


In [9]:
data = cwed4eta_process_json_file("DATA/LABEL_STUDIO/project-6-at-2025-03-31-11-50-c08f75da.json")


In [11]:
data

[{'text': 'The marine algal toxin domoic acid is an important threat to marine mammal health, and exposure can lead to both acute neurologic signs and a chronic epileptic syndrome in California sea lions (Zalophus californianus).',
  'entities': [{'text': 'marine algal toxin',
    'label': 'Chemical',
    'start': 4,
    'end': 22},
   {'text': 'domoic acid', 'label': 'Chemical', 'start': 23, 'end': 34},
   {'text': 'marine mammal health', 'label': 'Other', 'start': 61, 'end': 81},
   {'text': 'acute neurologic signs',
    'label': 'Disease',
    'start': 113,
    'end': 135},
   {'text': 'chronic epileptic syndrome',
    'label': 'Disease',
    'start': 142,
    'end': 168},
   {'text': 'California sea lions',
    'label': 'Organism',
    'start': 172,
    'end': 192},
   {'text': 'Zalophus californianus',
    'label': 'Organism',
    'start': 194,
    'end': 216}],
  'id': '10167-0'},
 {'text': 'Phenobarbital has been used for several decades to manage seizures, although reports are 

In [51]:
from dataset_processing import BIODIVNER_DIR, load_biodivner, transform_to_ner_format, ner_dataset_to_hf_format, generate_consistent_label_map, BIODIVNER_LABELS, biodivner_process_bio_documents

In [52]:
print("Converting from BIO tags to char spans.")
split_name = "train"
hf_name, safe_hf_name, output_dir = "BioDivNER", "BioDivNER", "PLOTS"


biodivner_structured_data = biodivner_process_bio_documents(
    data_dir=BIODIVNER_DIR, 
    labels_to_keep=BIODIVNER_LABELS, 
    split=[split_name]#, "validation", "test"]
    )

Converting from BIO tags to char spans.
Processing file: /home/p0l3/RAD/CWED4ETA/CWED4ETA/CWED4ETA/DATA/BiodivNER/train.csv...


In [53]:
transformed_dataset, tag_map = transform_to_ner_format(biodivner_structured_data, BIODIVNER_LABELS)

In [54]:
dataset_train = ner_dataset_to_hf_format(transformed_dataset, tag_map, test_size=0, val_size=0)

In [55]:
from EXPERIMENTS.calculate_hf_dataset_stats import process_dataset_split, plot_class_distribution

In [56]:
stats, counts = process_dataset_split(dataset_train["train"], list(generate_consistent_label_map(BIODIVNER_LABELS).keys()))

In [57]:
# Print to Console
print(f"{split_name:<12} | "
      f"{stats['Sentence Length (Mean)']:<10.2f} | "
      f"{stats['Entity Density (Avg entities/sent)']:<10.2f} | "
      f"{stats['Ratio (Median Len / Density)']:<10.2f} | "
      f"{stats['Ratio (Mean Len / Density)']:<10.2f} | "
      f"{stats['Span Length (Avg tokens/entity)']:<10.2f} | "
      f"{int(stats['Total Entities']):<10}")

# Generate Plot
plot_class_distribution(counts, split_name, hf_name, safe_hf_name, output_dir)

train        | 46.76      | 4.00       | 7.51       | 11.70      | 1.50       | 7665      
